# Snowflake CI/CD Notebook
**Account:** FJC24794 | **Repo:** github.com/hariopen75/snowflake-cicd

This notebook is the CI/CD runner. It is executed automatically:
- Every hour via Snowflake Task (`cicd_hourly_deploy`)
- On every GitHub push via GitHub Actions → Snowflake REST API

## Pipeline Steps
1. Validate environment
2. Fetch latest code from GitHub
3. Run schema migrations (SQL)
4. Run data quality tests
5. Deploy/refresh Dynamic Tables
6. Log results

In [ ]:
# ── CELL 1: Setup & imports
import snowflake.snowpark as snowpark
from snowflake.snowpark.context import get_active_session
from datetime import datetime

session = get_active_session()
print(f'Session: {session.get_current_database()}.{session.get_current_schema()}')
print(f'Role   : {session.get_current_role()}')
print(f'WH     : {session.get_current_warehouse()}')
print(f'Run at : {datetime.now()}')

PIPELINE_STATUS = []   # collect step results

In [ ]:
# ── CELL 2: Fetch latest from GitHub
try:
    session.sql('ALTER GIT REPOSITORY demo_db.analytics.snowflake_cicd_repo FETCH').collect()
    print('✓ Git repo synced to latest commit')

    # Get latest commit info
    files = session.sql("""
        SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED
        FROM DIRECTORY(@demo_db.analytics.snowflake_cicd_repo/branches/main/)
        ORDER BY LAST_MODIFIED DESC
    """).collect()
    print(f'  Files in repo: {len(files)}')
    for f in files[:5]:
        print(f'  {f[0]:50s} {f[2]}')
    PIPELINE_STATUS.append({'step': 'git_fetch', 'status': 'OK'})
except Exception as e:
    print(f'✗ Git fetch failed: {e}')
    PIPELINE_STATUS.append({'step': 'git_fetch', 'status': 'FAILED', 'error': str(e)})

In [ ]:
# ── CELL 3: Run schema migrations from Git repo
import os

migration_errors = []

# Read migration SQL files from the Git stage
try:
    migration_files = session.sql("""
        SELECT RELATIVE_PATH FROM DIRECTORY(
            @demo_db.analytics.snowflake_cicd_repo/branches/main/migrations/
        )
        WHERE RELATIVE_PATH LIKE '%.sql'
        ORDER BY RELATIVE_PATH   -- run in filename order (e.g. 001_, 002_)
    """).collect()
except:
    migration_files = []
    print('  No migrations/ directory found — skipping')

for row in migration_files:
    path = row[0]
    try:
        sql_text = session.sql(f"""
            SELECT $1 FROM @demo_db.analytics.snowflake_cicd_repo/branches/main/{path}
        """).collect()
        sql = ' '.join([r[0] for r in sql_text])

        # Execute each statement
        for stmt in sql.split(';'):
            stmt = stmt.strip()
            if stmt and not stmt.startswith('--'):
                session.sql(stmt).collect()
        print(f'  ✓ Migration: {path}')
    except Exception as e:
        print(f'  ✗ Migration FAILED: {path} → {e}')
        migration_errors.append({'file': path, 'error': str(e)})

status = 'OK' if not migration_errors else 'PARTIAL'
PIPELINE_STATUS.append({'step': 'migrations', 'status': status,
                         'ran': len(migration_files), 'errors': migration_errors})
print(f'Migrations: {len(migration_files)} files, {len(migration_errors)} errors')

In [ ]:
# ── CELL 4: Data Quality Tests
tests = []

def run_test(name, sql, expect_zero=True):
    """Run a data quality check. Pass if result == 0 (no violations)."""
    try:
        result = session.sql(sql).collect()
        val = result[0][0] if result else 0
        passed = (val == 0) if expect_zero else (val > 0)
        icon = '✓' if passed else '✗'
        print(f'  {icon} {name}: {val}')
        tests.append({'name': name, 'passed': passed, 'value': val})
        return passed
    except Exception as e:
        print(f'  ✗ {name}: ERROR → {e}')
        tests.append({'name': name, 'passed': False, 'error': str(e)})
        return False

print('Running data quality tests...')

# Test 1: No null customer IDs in orders
run_test('No null customer_id in orders',
    'SELECT COUNT(*) FROM demo_db.raw.orders WHERE customer_id IS NULL')

# Test 2: No negative amounts
run_test('No negative order amounts',
    'SELECT COUNT(*) FROM demo_db.raw.orders WHERE amount < 0')

# Test 3: No duplicate order IDs
run_test('No duplicate order_ids',
    'SELECT COUNT(*) FROM (SELECT order_id, COUNT(*) c FROM demo_db.raw.orders GROUP BY 1 HAVING c > 1)')

# Test 4: Orders table has data
run_test('Orders table not empty',
    'SELECT COUNT(*) FROM demo_db.raw.orders', expect_zero=False)

# Test 5: Referential integrity — all orders have a valid customer
run_test('All orders have valid customer',
    '''SELECT COUNT(*) FROM demo_db.raw.orders o
       WHERE NOT EXISTS (SELECT 1 FROM demo_db.raw.customers c WHERE c.customer_id = o.customer_id)''')

passed = sum(1 for t in tests if t.get('passed'))
total  = len(tests)
print(f'\nTests: {passed}/{total} passed')
PIPELINE_STATUS.append({'step': 'data_quality', 'status': 'OK' if passed == total else 'WARNINGS',
                         'passed': passed, 'total': total})

In [ ]:
# ── CELL 5: Deploy — Refresh Dynamic Tables + Tasks
deploys = []

def refresh_dynamic_table(fqn):
    try:
        session.sql(f'ALTER DYNAMIC TABLE {fqn} REFRESH').collect()
        print(f'  ✓ Refreshed: {fqn}')
        deploys.append({'object': fqn, 'status': 'OK'})
    except Exception as e:
        print(f'  ✗ Failed: {fqn} → {e}')
        deploys.append({'object': fqn, 'status': 'FAILED', 'error': str(e)})

print('Refreshing dynamic tables...')
refresh_dynamic_table('demo_db.analytics.customer_ltv')
refresh_dynamic_table('demo_db.analytics.daily_revenue')
refresh_dynamic_table('demo_db.analytics.customers_scd2')

print('\nResuming tasks...')
for task in ['cicd_hourly_deploy', 'load_orders_to_staging', 'refresh_customer_summary']:
    try:
        session.sql(f'ALTER TASK demo_db.analytics.{task} RESUME').collect()
        print(f'  ✓ Task resumed: {task}')
    except Exception as e:
        print(f'  - Task {task}: {e}')

PIPELINE_STATUS.append({'step': 'deploy', 'status': 'OK', 'objects': deploys})

In [ ]:
# ── CELL 6: Log pipeline results
import json

overall = 'SUCCESS' if all(s.get('status') in ('OK', 'WARNINGS') for s in PIPELINE_STATUS) else 'FAILED'

try:
    session.sql(f"""
        INSERT INTO demo_db.analytics.cicd_run_log
            (run_time, branch, notebook_path, triggered_by, status, error_message)
        SELECT
            CURRENT_TIMESTAMP(),
            'main',
            'notebooks/cicd_pipeline.ipynb',
            CURRENT_USER(),
            '{overall}',
            '{json.dumps(PIPELINE_STATUS)[:1000]}'
    """).collect()
    print(f'Pipeline logged: {overall}')
except Exception as e:
    print(f'Log write failed (table may not exist yet): {e}')

print('\n' + '='*50)
print(f'PIPELINE COMPLETE — Status: {overall}')
print('='*50)
for step in PIPELINE_STATUS:
    icon = '✓' if step.get('status') in ('OK','WARNINGS') else '✗'
    print(f'  {icon} {step["step"]:20s} → {step["status"]}')